# 07 — Model Selection & Cross-Validation

Estimating generalization performance and tuning hyperparameters without overfitting to the test set.

## cross_val_score (quick CV estimate)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(clf, X_train, y_train, cv=5, scoring="accuracy", n_jobs=-1)
scores.mean(), scores.std()


## cross_validate (multiple metrics + timing)

In [ ]:
from sklearn.model_selection import cross_validate

results = cross_validate(
    clf, X_train, y_train, cv=5,
    scoring=["accuracy", "precision_macro", "recall_macro", "f1_macro"],
    return_train_score=True,
)
pd.DataFrame(results)


## CV Splitting Strategies

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold, GroupKFold, TimeSeriesSplit, LeaveOneOut

KFold(n_splits=5, shuffle=True, random_state=42)                 # generic
StratifiedKFold(n_splits=5, shuffle=True, random_state=42)        # preserves class ratios — default for classifiers
GroupKFold(n_splits=5)                                             # keeps groups (e.g. same patient) in one fold only
TimeSeriesSplit(n_splits=5)                                        # respects temporal order, no shuffling
LeaveOneOut()                                                       # n folds of size 1, expensive but exhaustive


## GridSearchCV (exhaustive hyperparameter search)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_leaf": [1, 2, 4],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid, cv=5, scoring="f1_macro", n_jobs=-1, verbose=1
)
grid.fit(X_train, y_train)
grid.best_params_
grid.best_score_
grid.best_estimator_          # refit on full training set with best params (refit=True by default)
pd.DataFrame(grid.cv_results_)


## RandomizedSearchCV (sample from distributions, cheaper than full grid)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    "n_estimators": randint(50, 500),
    "max_depth": randint(3, 30),
    "max_features": uniform(0.1, 0.9),
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=50, cv=5, scoring="f1_macro", n_jobs=-1, random_state=42
)
random_search.fit(X_train, y_train)
random_search.best_params_


## Nested Cross-Validation (unbiased estimate when also tuning hyperparameters)

In [ ]:
from sklearn.model_selection import cross_val_score

inner_cv = GridSearchCV(RandomForestClassifier(), param_grid, cv=3, scoring="f1_macro")
outer_scores = cross_val_score(inner_cv, X_train, y_train, cv=5, scoring="f1_macro")
outer_scores.mean()   # unbiased performance estimate — tuning happens only inside each outer fold


## Learning Curves (diagnose over/underfitting vs. training set size)

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(), X_train, y_train, cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring="accuracy"
)
plt.plot(train_sizes, train_scores.mean(axis=1), label="train")
plt.plot(train_sizes, val_scores.mean(axis=1), label="validation")
plt.xlabel("training set size"); plt.legend()
plt.show()


## Validation Curves (diagnose over/underfitting vs. one hyperparameter)

In [ ]:
from sklearn.model_selection import validation_curve

param_range = [1, 5, 10, 20, 30, 50]
train_scores, val_scores = validation_curve(
    RandomForestClassifier(), X_train, y_train,
    param_name="max_depth", param_range=param_range, cv=5, scoring="accuracy"
)
plt.plot(param_range, train_scores.mean(axis=1), label="train")
plt.plot(param_range, val_scores.mean(axis=1), label="validation")
plt.xlabel("max_depth"); plt.legend()
plt.show()


## Common Gotchas
- Never tune hyperparameters using the final test set — use CV on the training set (or a separate validation set), reserve the test set for the very last, single evaluation.
- `StratifiedKFold` is used automatically by `cross_val_score`/`GridSearchCV` for classifiers — but explicit `KFold` is used for regressors; pass the splitter explicitly if unsure.
- `RandomizedSearchCV` with enough `n_iter` often finds comparable hyperparameters to `GridSearchCV` at a fraction of the compute cost.
- Time series data must use `TimeSeriesSplit`, never shuffled K-Fold, or you leak future information into training folds.